In [0]:
import requests
import pandas as pd

resp = requests.get(
    "https://archive-api.open-meteo.com/v1/archive",
    params={
        "latitude": 40.6413,
        "longitude": -73.7781,
        "start_date": "2025-06-01",
        "end_date": "2026-07-31",
        "hourly": "temperature_2m,relative_humidity_2m,precipitation,snowfall,"
                  "wind_speed_10m,wind_direction_10m,wind_gusts_10m,"
                  "cloud_cover,pressure_msl,weather_code",
        "temperature_unit": "fahrenheit",
        "wind_speed_unit": "mph",
        "precipitation_unit": "inch",
        "timezone": "America/New_York",
    },
    timeout=60,
)
resp.raise_for_status()

pdf = pd.DataFrame(resp.json()["hourly"])
pdf["time"] = pd.to_datetime(pdf["time"])

df = spark.createDataFrame(pdf)
df.write.mode("overwrite").saveAsTable("flight_delay_predictor_project.default.jfk_weather_history")
display(df)

In [0]:
%sql
CREATE OR REPLACE TABLE flight_delay_predictor_project.default.jfk_flights_with_weather_try_1 AS
WITH flights_parsed AS (
  SELECT
    *,
    -- Handle the common BTS date formats: "6/1/2025 12:00:00 AM", "2025-06-01", or an existing timestamp
    COALESCE(
      try_to_timestamp(CAST(FL_DATE AS STRING), 'M/d/yyyy h:mm:ss a'),
      try_to_timestamp(CAST(FL_DATE AS STRING), 'yyyy-MM-dd HH:mm:ss'),
      try_to_timestamp(CAST(FL_DATE AS STRING), 'yyyy-MM-dd')
    ) AS fl_date_ts,
    CAST(CRS_DEP_TIME AS INT) AS crs_dep_int
  FROM flight_delay_predictor_project.default.jfk_flight_data
),
flights_hourly AS (
  SELECT
    *,
    -- Hour bucket, e.g. 1435 -> 14:00 on that date (2400 rolls to the next day)
    timestampadd(HOUR, crs_dep_int DIV 100, fl_date_ts) AS dep_hour,
    -- Exact scheduled departure time, handy as a feature later
    timestampadd(MINUTE, crs_dep_int % 100,
                 timestampadd(HOUR, crs_dep_int DIV 100, fl_date_ts)) AS crs_dep_ts
  FROM flights_parsed
)
SELECT
  f.*,
  w.* EXCEPT (time)
FROM flights_hourly f
LEFT JOIN flight_delay_predictor_project.default.jfk_weather_history w
  ON f.dep_hour = w.time;

In [0]:
%sql
SELECT *
FROM flight_delay_predictor_project.default.jfk_flights_with_weather_try_1

In [0]:
%sql
SELECT cancelled, weather_delay, temperature_2m, relative_humidity_2m, precipitation, snowfall, wind_speed_10m, wind_direction_10m, wind_gusts_10m, cloud_cover, pressure_msl, weather_code
FROM flight_delay_predictor_project.default.jfk_flights_with_weather_try_1;

In [0]:
%sql
CREATE OR REPLACE TABLE flight_delay_predictor_project.default.first_prediction_try_1
SELECT cancelled, weather_delay, temperature_2m, relative_humidity_2m, precipitation, snowfall, wind_speed_10m, wind_direction_10m, wind_gusts_10m, cloud_cover, pressure_msl, weather_code
FROM flight_delay_predictor_project.default.jfk_flights_with_weather_try_1;

In [0]:
%pip install seaborn

In [0]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

In [0]:
df = spark.table('flight_delay_predictor_project.default.first_prediction_try_1').toPandas()

In [0]:
print(df.head())
print(len(df))

In [0]:
df.info()

In [0]:
from sklearn.model_selection import train_test_split

In [0]:
df["weather_delay"] = df["weather_delay"].fillna(0)

In [0]:
flights_1 = df[df['weather_delay'] != 0]
flights_2 = df[df['weather_delay'] == 0]

In [0]:
flights_1.shape

In [0]:
flights_2.shape

In [0]:
from sklearn.utils import resample

flights_1_upsampled = resample(flights_1,random_state=42,n_samples=101915,replace=True)
flights_upsampled = pd.concat([flights_1_upsampled,flights_2])

In [0]:
X = flights_upsampled.drop('weather_delay',axis=1)
y = flights_upsampled['weather_delay']

In [0]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30,random_state=42)

In [0]:
from sklearn.tree import DecisionTreeClassifier

In [0]:
dtree = DecisionTreeClassifier()

In [0]:
dtree.fit(X_train,y_train)

In [0]:
dtree.get_params()

In [0]:
predictions = dtree.predict(X_test)

In [0]:
from sklearn.metrics import classification_report,confusion_matrix

In [0]:
print(classification_report(y_test,predictions))

In [0]:
print(confusion_matrix(y_test,predictions))